# Load all required packages & Global Variables

In [ ]:
import polars as pl
from glob import glob
from os.path import join as here
import os
import xml.etree.ElementTree as ET
import yaml
from ultralytics import YOLO
import shutil
from pathlib import Path
import rasterio
import torch
from itertools import combinations
import optuna
import csv
from polars import selectors as cs
import logging
from ultralytics.utils import LOGGER
from datetime import datetime
from zoneinfo import ZoneInfo

import os
from anthropic import Anthropic
from dotenv import load_dotenv
import json

# turn off the per epoch run statistics
LOGGER.setLevel(logging.ERROR)

load_dotenv()
 
client = Anthropic(api_key = os.environ.get("claude"))  # reads ANTHROPIC_API_KEY
 
MODEL = "claude-fable-5"

# Compatibility Checks
Ensure that the GPU is recognized

In [ ]:
# check the pytorch version
torch.__version__

In [ ]:
# Ensure that torch cuda is true
assert torch.cuda.is_available() is True, "Cuda needs to be initialized"

# Define constanst & global variables

In [ ]:
# define what seed to use for random scattering of the folds
seed = 1

# define how many folds to use for stratified k_fold validation, run on the training data
folds = 3

# out of 100, define how much of the data to use for training, remainder will be used for validation
training_ratio = 80

# define how much of the data should be background (images with no labels) versus labeled images
# this value can range between 0 and 100
# general recommendation is to have it between 5 and 10%
background_ratio = 5

# define the folder path extension splitter, \\ for windows and / for mac
splitter = "\\"

# define the path to where the map_grid folders are contained
data_path = here("Data", "Export_Training_Data_PeachandEmptyTiles", "Export_Training_Data_PeachandEmptyTiles")

# ensure that all values are ints
assert all(isinstance(x, int) for x in (seed, folds, training_ratio)), "All three variables must be integers"

# Image Downsampling

Background images are images without any labels. Whereas it is useful to have some background images to improve performance, you generally do not want background images to be more than 5-10% of the dataset.

In [ ]:
# define a function to load all of the tif images in each folder
def all_tifs(path):
    return glob(here(path, "images", "*.tif"))

# define a function to find whether a label file exists for the image
def is_image_not_background(path):
    return os.path.exists(path.replace("images", "labels").replace("tif", "xml"))

In [ ]:
sample = pl.DataFrame(
    # load the file paths ensuring only to load folders
    {"paths": [path for path in glob(here(data_path, "*")) if os.path.isdir(path)]}
    ).with_columns(
        # subtract the final file path
        pl.col("paths").str.split(splitter).list.last().alias("area_grid")
    ).with_columns(
        # make a temporary list to separate the area and grids
        pl.col("area_grid").str.split("_").alias("temp")
    ).with_columns(
        # extract the area and grid
        pl.col("temp").list.get(0).alias("area"),
        pl.col("temp").list.get(2).alias("grid")
    ).drop("temp"
    ).with_columns(
        # find all of the tifs contained within the folder
        pl.col("paths").map_elements(all_tifs, return_dtype = pl.List(pl.String)).alias("tifs")
    ).explode("tifs", empty_as_null = True
    ).with_columns(
        # detect if there is a label file for the image
        pl.col("tifs").map_elements(is_image_not_background, return_dtype = pl.Boolean).alias("not_background")
    ).with_columns(
        # calculate the % of background images per area
        (pl.col("not_background").not_().sum().over("area")/pl.len().over("area")).alias("background %")
    )


sample.unique("area")

In [ ]:
# calculate the % of background images to remove to make the total 
background_cutoff = (sample.filter(~pl.col("not_background")).height - (background_ratio/100)*sample.height)/sample.filter(~pl.col("not_background")).height

# randomly select which background images to remove, done over each area to preserve stratification
background = sample.filter(~pl.col("not_background")).sample(
        # shuffle all of the rows in preparation for the user defined data split
        # this randomizes the data
        fraction=1, shuffle=True, seed=seed
    ).with_columns(
        # over each area, generate a column of the number of grids
        pl.int_range(pl.len()).over("area").alias("temp"),
        # define the cutoff for each to perform the split
        (pl.len() *  background_cutoff).over("area").ceil().cast(pl.Int128).alias("split_cutoff")
    ).with_columns(
        # assign each grid either training or validation based on the user defined split
        pl.when(pl.col("temp") > pl.col("split_cutoff")).then(True).otherwise(False).alias("keep")
    ).drop(["split_cutoff", "temp"]
    )

background.filter(pl.col("keep")).with_columns(
    # calculate the area composition of the kept data
    (pl.len().over("area")/pl.len()).alias("area %")
).unique("area")


In [ ]:
# define a function to convert tif files we do not want to dif files
def tif2dif(path):
    # generate the path file
    file_path = Path(path)
    file_path.rename(file_path.with_suffix(".dif"))


# WARNING
# Do not run this cell unless you change the files ending with the .dif extension back to .tif since this code will continously downsample the images
# all of the images in the Data folder have already been downsampled

if False:
    background.filter(~pl.col("keep")
    ).with_columns(pl.col("tifs").map_elements(tif2dif, return_dtype = pl.Null).alias("temp")).drop("temp")


# Generate Manifest & Splits

The first split is a random selection of data from each area depending on how the user sets the `training` variable. For example, if the user specifies 80, then rounded upwards depending on the length of the dataset 80% of the data would be assigned as training and the remainder as validation data.

The stratified K fold was then applied to the training data. The stratified portion to refers to the practice that a representative sample of each class, in this instance area left, middle, and right, are all equally represented in each of the random folds. To achieve this, over each area, we assigned each row a fold depending on however many the user defines, the folds of each area were then combined to form a stratified fold containing an equal random portion of each area.

https://www.tandfonline.com/doi/abs/10.1080/10106049.2019.1704070

In [ ]:
# given a path to an esri Pascal training data into a polars dataframe manifest of the data

manifest = pl.DataFrame(
    # load the file paths ensuring only to load folders
    {"paths": [path for path in glob(here(data_path, "*")) if os.path.isdir(path)]}
    ).with_columns(
        # subtract the final file path
        pl.col("paths").str.split(splitter).list.last().alias("area_grid")
    ).with_columns(
        # make a temporary list to separate the area and grids
        pl.col("area_grid").str.split("_").alias("temp")
    ).with_columns(
        # extract the area and grid
        pl.col("temp").list.get(0).alias("area"),
        pl.col("temp").list.get(2).alias("grid")
    ).drop("temp"
    ).sample(
        # shuffle all of the rows in preparation for the user defined data split
        # this randomizes the data
        fraction=1, shuffle=True, seed=seed
    ).with_columns(
        # over each area, generate a column of the number of grids
        pl.int_range(pl.len()).over("area").alias("temp"),
        # define the cutoff for each to perform the split
        (pl.len() * (training_ratio / 100)).over("area").ceil().cast(pl.Int128).alias("split_cutoff")
    ).with_columns(
        # assign each grid either training or validation based on the user defined split
        pl.when(pl.col("temp") < pl.col("split_cutoff")).then(pl.lit("training")).otherwise(pl.lit("validation")).alias("tra_or_val")
    ).drop(["split_cutoff", "temp"]
    ).with_columns(
        # calculate the overall ratio of training data across all areas
        (pl.col("tra_or_val").eq("training").sum()/pl.len()).alias("training_ratio"),
        (pl.col("tra_or_val").eq("training").sum().over("area")/pl.len().over("area")).alias("training_ratio_per_area")
    )

# select the training data, assign each of the randomized rows a fold
train = manifest.filter(pl.col("tra_or_val").eq("training")).with_columns(
    # over every area assign each column to a fold
    pl.int_range(pl.len()).over("area").mod(folds).alias("fold")
).with_columns(
    # calculate the percentage representation of each area in each fold
    (pl.len().over(["fold", "area"]) / pl.len().over("fold")).alias("area_fold_ratio")
)

# select the validation data (test data)
valid = manifest.filter(~pl.col("tra_or_val").eq("training"))


# Convert to YOLO Compatible Format
The Pascal training data does not automatically work out of the Box with Yolo models, to address this we need to shift all of the files into a YOLO compatible folder structure and convert the types to YOLO compatible types.

## Generate run folder

In order to train the YOLO model, we need to create a folder that contains all of the images and labels to use for training and validation data.

It has to follow this general format:

```
run/
├── images/
│   ├── train/  (contains 80% of your .tif files)
│   └── val/    (contains the remaining 20% of .tif files)
└── labels/
    ├── train/  (contains the .txt files matching the train .tifs)
    └── val/    (contains the .txt files matching the val .tifs)

```

In [ ]:
# define a helper function that makes the YOLO compatible folder format
def make_fold_folder(run_folder):

    # delete and create a new folder
    if os.path.exists(here(run_folder)):
        shutil.rmtree(here(run_folder))
    os.makedirs(here(run_folder))

    # generate the train and val folder, YOLO model assumes the label placement from image folder path
    image_train_path = here(run_folder, "images", "train")
    image_val_path = here(run_folder, "images", "val")

    label_train_path = here(run_folder, "labels", "train")
    label_val_path = here(run_folder, "labels", "val")

    for path in [image_train_path, image_val_path, label_train_path, label_val_path]:
        os.makedirs(path)
    
    return image_train_path, image_val_path, label_train_path, label_val_path

## Define function that moves Tiff files
This function converts any Tiff from 4 to 3 bands and moves them to the new folder

In [ ]:

# converts tiff from 4 bands to 3 bands
def pl_move_tiff(struct, image_train_path, image_val_path):
    
    if struct["train"] is None:
        return None
    elif struct["train"] is True:
        for file in glob(here(struct['paths'], "images", "*.tif")):
            with rasterio.open(file) as src:
                band1 = src.read(1)
                band2 = src.read(2)
                band3 = src.read(3)

                kwargs = src.meta.copy()
                # Remove the spatial coordinate system tags
                if 'crs' in kwargs:
                    del kwargs['crs']
                if 'transform' in kwargs:
                    del kwargs['transform']

                # Update to 3 bands
                kwargs.update({"count": 3})

                with rasterio.open(here(image_train_path, struct["area_grid"] + os.path.basename(file)), "w", **kwargs) as dst:
                    dst.write(band1, 1)
                    dst.write(band2, 2)
                    dst.write(band3, 3)


    elif struct["train"] is False:
        for file in glob(here(struct['paths'], "images", "*.tif")):
            with rasterio.open(file) as src:
                band1 = src.read(1)
                band2 = src.read(2)
                band3 = src.read(3)

                kwargs = src.meta.copy()
                # Remove the spatial coordinate system tags
                if 'crs' in kwargs:
                    del kwargs['crs']
                if 'transform' in kwargs:
                    del kwargs['transform']

                # Update to 3 bands
                kwargs.update({"count": 3})

                with rasterio.open(here(image_val_path, struct["area_grid"] + os.path.basename(file)), "w", **kwargs) as dst:
                    dst.write(band1, 1)
                    dst.write(band2, 2)
                    dst.write(band3, 3)

## Define a function to move the xml files and convert them
Define a function that converts the xml file to YOLO compatible txt format and moves the function

In [ ]:
# Define function to take an arbitrary xml file and export as txt
def xml2txt(xml_path:str):
    """
    Given the path to an XML Pascal file, return the file converted to txt format
    """
    # parse the XML file into a root object
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # get the size dimensions
    size_node = root.find("size")
    img_width = float(size_node.find("width").text)
    img_height = float(size_node.find("height").text)

    # create the empty string to write to the txt file
    txt = ""

    # loop through each peach
    for obj in root.findall("object"):
        # Get absolute bounding box coordinates
        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)
        
        #  Do the math (Absolute -> Relative)
        abs_width = xmax - xmin
        abs_height = ymax - ymin
        abs_x_center = xmin + (abs_width / 2.0)
        abs_y_center = ymin + (abs_height / 2.0)
        
        norm_x = abs_x_center / img_width
        norm_y = abs_y_center / img_height
        norm_w = abs_width / img_width
        norm_h = abs_height / img_height
        
        # append the line to the txt
        txt += f"{"0"} {norm_x:.6f} {norm_y:.6f} {norm_w:.6f} {norm_h:.6f}\n"
    
    return txt


def pl_move_xml(struct, label_train_path, label_val_path):
    
    if struct["train"] is None:
        return None
    elif struct["train"] is True:
        for file in glob(here(struct['paths'], "labels", "*.xml")):
            # extract the base file
            base_file = struct["area_grid"] + Path(file).stem +'.txt'
            with open(here(label_train_path, base_file), 'w') as f:
                f.write(xml2txt(file))

    elif struct["train"] is False:
        for file in glob(here(struct['paths'], "labels", "*.xml")):
            # extract the base file
            base_file = struct["area_grid"] + Path(file).stem +'.txt'
            with open(here(label_val_path, base_file), 'w') as f:
                f.write(xml2txt(file))


## Define a function to shift everything over

In [ ]:
# define a function to convert all of the training and validation files to their proper format and structure
def shift(train:pl.DataFrame, image_train_path, image_val_path, label_train_path, label_val_paths, train_folds, validate_folds):
    train.with_columns(
        # detect which folds to train versus validate based on the user input
        pl.when(pl.col("fold").is_in(train_folds)).then(True).when(pl.col("fold").is_in(validate_folds)).then(False).otherwise(None).alias("train")
    ).with_columns(
        # move the tiff and xml data
        pl.struct(["train", "paths", "area_grid"]).map_elements(
            lambda x: pl_move_xml(x, label_train_path, label_val_paths), return_dtype=pl.Null, strategy='threading'
        ).alias("temp1"),
        pl.struct(["train", "paths", "area_grid"]).map_elements(
            lambda x: pl_move_tiff(x, image_train_path, image_val_path), return_dtype=pl.Null, strategy = 'threading'
        ).alias("temp2")
    )

# Generate YAML file
Generate a YAML file helper function so that the YOLO model knows what to use for its training and validation data

In [ ]:
# define a function to create a yaml for the YOLO model, this tells it what data it has to train and validate on

def make_yaml(run_folder, image_train_path, image_val_path):
    yolo_config = {
        "train": image_train_path,
        "val": image_val_path,
        "ch": 3,           # Number of channels for your TIFFs
        "nc": 1,           # Number of classes
        "names": [         # List of class names (must match your 0, 1, 2 text file IDs)
            "peach" # define that all of the labels are peaches
        ]
    }

    # 2. Write the dictionary to a YAML file
    with open(f"{run_folder}.yaml", "w") as yaml_file:
        # default_flow_style=False ensures it prints cleanly with line breaks
        # sort_keys=False ensures the keys print in the order you defined them
        yaml.dump(yolo_config, yaml_file, default_flow_style=False, sort_keys=False)

    print(f"{run_folder}.yaml generated successfully!")

# Perform stratified 3-fold cross validation split
Split the xml and tiff images equally and randomly across each area into three folders and generate 3 yamls for the code to run from

In [ ]:
# loop through and generate the yaml, training, and validation data splits for each 3 fold variation
fold_range = list(range(folds))

# create a list to store the yaml names
yamls = []
runs = []

for combo in combinations(fold_range, 2):

    # define the folds for each unique iteration
    train_folds = combo
    # anything not used in training is used for validation
    validate_folds = [i for i in fold_range if i not in combo ]

    # generate the run folder name
    run_folder = "train_folds" + "_" + "_".join([str(i) for i in train_folds]) + "_valid_folds_" + "".join([str(i) for i in validate_folds])

    runs.append(run_folder)

    # generate the folder, receive the names
    image_train_path, image_val_path, label_train_path, label_val_path = make_fold_folder(run_folder)

    # copy over the data to the folders
    shift(train, image_train_path, image_val_path, label_train_path, label_val_path, train_folds, validate_folds)

    # make the yaml
    make_yaml(run_folder, image_train_path, image_val_path)

    # append the yaml
    yamls.append(f"{run_folder}.yaml")

# Define helper functions to move test data
We also need helper functions to shift over all of the test data to perform the final data on the models when they are trained on all of the training data

In [ ]:
# create the folder for the test data
def make_test_folder(test_folder):

    # delete and create a new folder
    if os.path.exists(here(test_folder)):
        shutil.rmtree(here(test_folder))
    os.makedirs(here(test_folder))

    # generate the train and val folder, YOLO model assumes the label placement from image folder path
    image_test = here(test_folder, "images")

    label_test = here(test_folder, "labels")

    for path in [image_test, label_test]:
        os.makedirs(path)
    
    return image_test, label_test

In [ ]:
# define a new shift function for the test data
def shift_test(train:pl.DataFrame, image_test, label_test):
    train.with_columns(
        pl.lit(False).alias("train")
    ).with_columns(
        # move the tiff and xml data
        pl.struct(["train", "paths", "area_grid"]).map_elements(
            lambda x: pl_move_xml(x, label_test, label_test), return_dtype=pl.Null, strategy='threading'
        ).alias("temp1"),
        pl.struct(["train", "paths", "area_grid"]).map_elements(
            lambda x: pl_move_tiff(x, image_test, image_test), return_dtype=pl.Null, strategy = 'threading'
        ).alias("temp2")
    )

In [ ]:
# make a test data yaml file
def make_test_yaml(test_folder, image_test):
    yolo_config = {
        # YOLO requires a train and validate no matter what so hard code
        "train": here("TPE_all_train", "images", "train"),
        "val": here("TPE_all_train", "images", "train"),
        "test": image_test,
        "ch": 3,           # Number of channels for your TIFFs
        "nc": 1,           # Number of classes
        "names": [         # List of class names (must match your 0, 1, 2 text file IDs)
            "peach"
        ]
    }

    # 2. Write the dictionary to a YAML file
    with open(f"{test_folder}.yaml", "w") as yaml_file:
        # default_flow_style=False ensures it prints cleanly with line breaks
        # sort_keys=False ensures the keys print in the order you defined them
        yaml.dump(yolo_config, yaml_file, default_flow_style=False, sort_keys=False)

    print(f"{test_folder}.yaml generated successfully!")



# Base model with no hyperparameters
To have a fair benchmark for the LLM and TPE, we need to first run the base YOLO model with default hyperparameters as a control

Note, YOLO requires a valid path for both the training and validation datasets, but since we are training the YOLO model on all of the training data and testing it on the test data, there is no validation data. To get around this, manually edit the YAML file so that the validation path is the same as the training path.

In [ ]:
# generate the run folder name
None_run_folder = "None_all_train"

# generate the folder, receive the names
image_train_path, image_val_path, label_train_path, label_val_path = make_fold_folder(None_run_folder)

# copy over the data to the folders
shift(train, image_train_path, image_val_path, label_train_path, label_val_path, [0,1,2], [])

# make the yaml
make_yaml(None_run_folder, image_train_path, image_val_path)

# append the yaml
None_yaml = f"{None_run_folder}.yaml"

# NOTE: MANUALLY HAVE TO EDIT YAML TO HAVE SAME PATH FOR TRAIN AND VALIDATION SO THAT YOLO IS SATISFIED THAT VALIDATION PATH IS VALID EVEN IF IT DOES NOT USE IT
  

In [ ]:
None_parameters =  {
        'lr0': 0.00214018635106669,
        'lrf': 0.7748126790331027,
        'momentum': 0.8421132632128773, 
        'weight_decay': 0.00039198918876708825,
        'warmup_epochs': 0.9039427650470803,
        'warmup_momentum': 0.7474088093561146,
        "val": False
    }

# turn on the yolo print statements
LOGGER.setLevel(logging.INFO)

model = YOLO("yolo26n.pt")

results = model.train(
    data=None_yaml,
    epochs=100,
    imgsz=640,
    batch=24,
    device=0,
    optimizer="SGD",
    seed=seed,
    patience=50,
    deterministic=True,
    project="runs/None",
    name=f"None_parameters",
    verbose=True,
    plots=False,
    **None_parameters
)

In [ ]:
model = YOLO(here("runs","detect", "runs" "None", "None_parameters", "weights", "last.pt"))

metrics_None = model.val(
    data="TEST.yaml",
    split="test",
    imgsz=640,
    batch=24,
    device=0,
    plots=True,
    save_json=True,
    save_txt=True,
    save_conf=True,
)

In [ ]:
box_None = metrics_None.box

print("Mean precision:", box_None.mp)
print("Mean recall:", box_None.mr)
print("mAP50:", box_None.map50)
print("mAP75:", box_None.map75)
print("mAP50-95:", box_None.map)
print("Fitness:", metrics_None.fitness)

# Optuna TPE Study Training
Define a harness for Optuna's TPE method to control the YOLO model hyperparameters

In [ ]:
# harness wrapper function so Optuna can pass trial data to and from YOLO
def objective(trial, study_name:str):
    # define the fixed hyperparameters and the ranges from which Optuna can adjust things
    params = {
        'lr0': 0.00214018635106669,
        'lrf': 0.7748126790331027,
        'momentum': 0.8421132632128773, 
        'weight_decay': 0.00039198918876708825,
        'warmup_epochs': 0.9039427650470803,
        'warmup_momentum': 0.7474088093561146,
        "hsv_h": trial.suggest_float("hsv_h", 0.0, 0.1),
        "hsv_s": trial.suggest_float("hsv_s", 0.0, 0.9),
        "hsv_v": trial.suggest_float("hsv_v", 0.0, 0.9),
        "degrees": trial.suggest_float("degrees", 0.0, 45.0),
        "translate": trial.suggest_float("translate", 0.0, 0.9),
        "scale": trial.suggest_float("scale", 0.0, 0.95),
        "shear": trial.suggest_float("shear", 0.0, 10.0),
        "perspective": trial.suggest_float("perspective", 0.0, 0.001),
        "flipud": trial.suggest_float("flipud", 0.0, 1.0),
        "fliplr": trial.suggest_float("fliplr", 0.0, 1.0),
        "bgr": trial.suggest_float("bgr", 0.0, 1.0),
        "mosaic": trial.suggest_float("mosaic", 0.0, 1.0),
        "mixup": trial.suggest_float("mixup", 0.0, 1.0),
        "cutmix": trial.suggest_float("cutmix", 0.0, 1.0),
        "copy_paste": trial.suggest_float("copy_paste", 0.0, 1.0),
        "close_mosaic": trial.suggest_int("close_mosaic", 0, 10),
    }

    # define what metrics to retain
    metric_keys = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
    "fitness",
]

    # define a dictionary to store the metrics we retain
    fold_metrics = {key: [] for key in metric_keys}

    # loop through each data fold
    for fold_i, (yaml_path, run_name) in enumerate(zip(yamls, runs)):
        # define the yolo model
        model = YOLO("yolo26n.pt")

        # train the yolo model with the Optuna determined hyperparameters
        results = model.train(
            data=yaml_path,
            epochs=100,
            imgsz=640,
            batch=24,
            device=0,
            optimizer="SGD",
            seed=seed,
            patience=50,
            deterministic=True,
            project="runs/hpo",
            name=f"trial_{trial.number}_{run_name}",
            verbose=False,
            plots=False,
            **params,
        )

        # print the results with flush so it resets each time
        map50 = results.results_dict["metrics/mAP50(B)"]
        map5095 = results.results_dict["metrics/mAP50-95(B)"]
        precision = results.results_dict["metrics/precision(B)"]
        recall = results.results_dict["metrics/recall(B)"]

        print(
            f"trial={trial.number} fold={fold_i} "
            f"mAP50={map50:.4f} "
            f"mAP50-95={map5095:.4f} "
            f"precision={precision:.4f} "
            f"recall={recall:.4f}",
            flush=True,
        )

        # log the data
        log_path = Path(f"hpo_live_metrics_{study_name}.csv")
        write_header = not log_path.exists()

        with open(log_path, "a", newline="") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=[
                    "trial",
                    "fold",
                    "run_name",
                    "map50",
                    "map50_95",
                    "precision",
                    "recall",
                    "fitness",
                ],
            )

            if write_header:
                writer.writeheader()

            writer.writerow({
                "trial": trial.number,
                "fold": fold_i,
                "run_name": run_name,
                "map50": map50,
                "map50_95": map5095,
                "precision": precision,
                "recall": recall,
                "fitness": results.results_dict["fitness"],
            })

        for key in metric_keys:
            fold_metrics[key].append(results.results_dict[key])

        current_mean_map5095 = (
            sum(fold_metrics["metrics/mAP50-95(B)"])
            / len(fold_metrics["metrics/mAP50-95(B)"])
        )

        trial.report(current_mean_map5095, step=fold_i)

        trial.set_user_attr("partial_fold_metrics", fold_metrics)

    # mean each metric between all three runs
    mean_metrics = {
        key: sum(values) / len(values)
        for key, values in fold_metrics.items()
    }

    trial.set_user_attr("fold_metrics", fold_metrics)
    trial.set_user_attr("mean_metrics", mean_metrics)

    # return the 3-fold average mAP50-95 
    return mean_metrics["metrics/mAP50-95(B)"]

In [ ]:
# run the TPE study for 30 trials
study_name = "TPE"

study = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=seed),
    pruner=optuna.pruners.NopPruner(),
    storage="sqlite:///peach_hpo.db",
    load_if_exists=True,
)

study.optimize(lambda x: objective(x, study_name=study_name), n_trials=30)

In [ ]:
# print out the best score and hyperparameters from the trials
study = optuna.load_study(
    study_name="TPE",
    storage="sqlite:///peach_hpo.db",
)

print("Best score:", study.best_value)
print("Best hyperparameters:")
print(study.best_params)

# Optuna TPE Study Testing
Note, YOLO requires a valid path for both the training and validation datasets, but since we are training the YOLO model on all of the training data and testing it on the test data, there is no validation data. To get around this, manually edit the YAML file so that the validation path is the same as the training path.

In [ ]:


# generate the run folder name
TPE_run_folder = "TPE_all_train"

# generate the folder, receive the names
image_train_path, image_val_path, label_train_path, label_val_path = make_fold_folder(TPE_run_folder)

# copy over the data to the folders
shift(train, image_train_path, image_val_path, label_train_path, label_val_path, [0,1,2], [])

# make the yaml
make_yaml(TPE_run_folder, image_train_path, image_val_path)

# append the yaml
TPE_yaml = f"{TPE_run_folder}.yaml"

# NOTE: MANUALLY HAVE TO EDIT YAML TO HAVE SAME PATH FOR TRAIN AND VALIDATION SO THAT YOLO IS SATISFIED THAT VALIDATION PATH IS VALID EVEN IF IT DOES NOT USE IT
    

In [ ]:
# load the best hyper parameters
TPE_study = optuna.load_study(
    study_name="TPE",
    storage="sqlite:///peach_hpo.db",
)

TPE_parameters = TPE_study.best_params

In [ ]:
# combine the TPE hyperparameters with the fixed parameters
full_TPE_parameters =  {
        'lr0': 0.00214018635106669,
        'lrf': 0.7748126790331027,
        'momentum': 0.8421132632128773, 
        'weight_decay': 0.00039198918876708825,
        'warmup_epochs': 0.9039427650470803,
        'warmup_momentum': 0.7474088093561146,
    } | TPE_parameters

full_TPE_parameters

In [ ]:
# turn on the yolo print statements
LOGGER.setLevel(logging.INFO)

model = YOLO("yolo26n.pt")

# ensure validation is disabled for this run
train_args = {
    **full_TPE_parameters,
    "val": False,
}

# train the model with the defined TPE hyperparameters on all of the training data
results = model.train(
    data=TPE_yaml,
    epochs=100,
    imgsz=640,
    batch=24,
    device=0,
    optimizer="SGD",
    seed=seed,
    patience=50,
    deterministic=True,
    project="runs/all_TPE",
    name=f"TPE_full_parameters",
    verbose=True,
    plots=False,
    **train_args,
)


In [ ]:
# move the test data into a compatible folder

test_folder = "TEST"

image_test, label_test = make_test_folder(test_folder)

shift_test(valid, image_test, label_test)

make_test_yaml(test_folder, image_test)

In [ ]:
# load the trained YOLO model
model = YOLO(here("runs", "detect", "runs", "all_TPE", "TPE_full_parameters", "weights", "last.pt"))

metrics_TPE = model.val(
    data="TEST.yaml",
    split="test",
    imgsz=640,
    batch=24,
    device=0,
    plots=True,
    save_json=True,
    save_txt=True,
    save_conf=True,
)

In [ ]:
box_TPE = metrics_TPE.box

print("Mean precision:", box_TPE.mp)
print("Mean recall:", box_TPE.mr)
print("mAP50:", box_TPE.map50)
print("mAP75:", box_TPE.map75)
print("mAP50-95:", box_TPE.map)
print("Fitness:", metrics_TPE.fitness)

# LLM Study Testing
This is the Autoresearch portion of the code where we allow the LLM to adjust the YOLO model hyperparameters

In [ ]:
# define the hyperparameter ranges and their default values for the LLM
SEARCH_SPACE = {
    "hsv_h":        ("float", 0.0, 0.1),
    "hsv_s":        ("float", 0.0, 0.9),
    "hsv_v":        ("float", 0.0, 0.9),
    "degrees":      ("float", 0.0, 45.0),
    "translate":    ("float", 0.0, 0.9),
    "scale":        ("float", 0.0, 0.95),
    "shear":        ("float", 0.0, 10.0),
    "perspective":  ("float", 0.0, 0.001),
    "flipud":       ("float", 0.0, 1.0),
    "fliplr":       ("float", 0.0, 1.0),
    "bgr":          ("float", 0.0, 1.0),
    "mosaic":       ("float", 0.0, 1.0),
    "mixup":        ("float", 0.0, 1.0),
    "cutmix":       ("float", 0.0, 1.0),
    "copy_paste":   ("float", 0.0, 1.0),
    "close_mosaic": ("int",   0,   10),
}
 
BASELINE_PARAMS = {
    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4, "degrees": 0.0,
    "translate": 0.1, "scale": 0.5, "shear": 0.0, "perspective": 0.0,
    "flipud": 0.0, "fliplr": 0.5, "bgr": 0.0, "mosaic": 1.0,
    "mixup": 0.0, "cutmix": 0.0, "copy_paste": 0.0, "close_mosaic": 10,
}

In [ ]:
# Build the Anthropic compatible schema
def build_schema():
    hp_props = {}
    for name, (kind, lo, hi) in SEARCH_SPACE.items():
        hp_props[name] = {
            "type": "object",
            "properties": {
                "justification": {"type": "string"},
                "value": {
                    "type": "integer" if kind == "int" else "number",
                     "description": f"Allowed range: {lo} to {hi} inclusive.",
                },
            },
            "required": ["justification", "value"],
            "additionalProperties": False,
        }
    return {
        "type": "object",
        "properties": {
            "analysis": {
                "type": "string",
                "description": (
                    "2-5 sentence strategy summary, written before any "
                    "hyperparameter values are chosen."
                ),
            },
            "hyperparameters": {
                "type": "object",
                "properties": hp_props,
                "required": list(SEARCH_SPACE.keys()),
                "additionalProperties": False,
            },
        },
        "required": ["analysis", "hyperparameters"],
        "additionalProperties": False,
    }
 
SCHEMA = build_schema()

In [ ]:
# load in the prompts
base_prompt = open(here("prompts", "base_prompt.txt")).read()
first_loop_prompt = open(here("prompts", "first_loop.txt")).read()
loop_prompt = open(here("prompts", "loop.txt")).read()

In [ ]:
# define the function to call the model
def call_fable_5(prompt: str) -> dict:
    response = client.messages.create(
        model=MODEL,
        max_tokens=16000,  # covers thinking + the JSON payload
        thinking={"type": "adaptive"},
        messages=[{"role": "user", "content": prompt}],
        output_config={
            "format": {"type": "json_schema", "schema": SCHEMA},
        },
    )
    text_block = next(b for b in response.content if b.type == "text")
    return json.loads(text_block.text)

In [ ]:
# define the function for the LLM to harness into the YOLO model
def LLM_objective(trial:int, hpo:dict):
    params = {
        'lr0': 0.00214018635106669,
        'lrf': 0.7748126790331027,
        'momentum': 0.8421132632128773, 
        'weight_decay': 0.00039198918876708825,
        'warmup_epochs': 0.9039427650470803,
        'warmup_momentum': 0.7474088093561146,
        **hpo
    }

    metric_keys = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
    "fitness",
]

    fold_metrics = {key: [] for key in metric_keys}

    for fold_i, (yaml_path, run_name) in enumerate(zip(yamls, runs)):
        model = YOLO("yolo26n.pt")

        results = model.train(
            data=yaml_path,
            epochs=100,
            imgsz=640,
            batch=24,
            device=0,
            optimizer="SGD",
            seed=seed,
            patience=50,
            deterministic=True,
            project="runs/LLM",
            name=f"trial_{trial}_{run_name}",
            verbose=False,
            plots=False,
            **params,
        )

        # print the results with flush so it resets each time
        map50 = results.results_dict["metrics/mAP50(B)"]
        map5095 = results.results_dict["metrics/mAP50-95(B)"]
        precision = results.results_dict["metrics/precision(B)"]
        recall = results.results_dict["metrics/recall(B)"]

        print(
            f"trial={trial} fold={fold_i} "
            f"mAP50={map50:.4f} "
            f"mAP50-95={map5095:.4f} "
            f"precision={precision:.4f} "
            f"recall={recall:.4f}",
            flush=True,
        )

        log_path = Path(f"LLM_values.csv")
        write_header = not log_path.exists()

        with open(log_path, "a", newline="") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=[
                    "trial",
                    "fold",
                    "run_name",
                    "map50",
                    "map50_95",
                    "precision",
                    "recall",
                    "fitness",
                ],
            )

            if write_header:
                writer.writeheader()

            writer.writerow({
                "trial": trial,
                "fold": fold_i,
                "run_name": run_name,
                "map50": map50,
                "map50_95": map5095,
                "precision": precision,
                "recall": recall,
                "fitness": results.results_dict["fitness"],
            })

        for key in metric_keys:
            fold_metrics[key].append(results.results_dict[key])





    mean_metrics = {
        key: sum(values) / len(values)
        for key, values in fold_metrics.items()
    }


    return pl.DataFrame(
        {   "trial": trial,
            "mean_mAP_50_95": mean_metrics["metrics/mAP50-95(B)"]
        }
    )

In [ ]:
def return_hpo(df: pl.DataFrame, trial:int):
    """
    Returns the hpo dict given the return df from the LLM
    """
    return df.filter(pl.col("trial").eq(trial)).select([cs.ends_with("_value")]).rename({
        col: col.replace("_value", "")
        for col in df.columns
    }).to_dicts()[0]

In [ ]:
def load_hpos(trial: int, LLM_HPO_path: Path):
    """
    If a prior run exists for this trial in the HPO dataset, load it
    """

    if LLM_HPO_path.exists():
        hpo = pl.read_csv(LLM_HPO_path)

        if trial in hpo["trial"].to_list():
            print("loading historic hpo")

            return return_hpo(hpo, trial)
        
        else:
            return None
    
    return None


In [ ]:
def load_mAPs(trial: int, LLM_mAP_path: Path):
    """
    If a prior run exists for this trial in the HPO dataset, load it
    """

    if LLM_mAP_path.exists():
        mAP = pl.read_csv(LLM_mAP_path)

        if trial in mAP["trial"].to_list():
            print("mAP already exists")

            return "mAP already exists, skip"
        
        else:
            return None
    
    return None

In [ ]:
def upsert_unique_csv(df: pl.DataFrame, csv_path) -> None:
    """
    Saves or concats a polars dataframe to the existing csv path
    """
    csv_path = Path(csv_path)

    if not csv_path.exists():
        df.write_csv(csv_path)
        return

    existing = pl.read_csv(csv_path)

    combined = pl.concat([existing, df], how="vertical_relaxed")

    unique = combined.unique(maintain_order=True)

    unique.write_csv(csv_path)

In [ ]:
# define a loop prompt function that provides in the trial parameters and run history to the LLM
def populate_loop_prompt(trial: int, LLM_HPO_path, LLM_mAP_path):
    # load the two dataframes storing the average mAP50 and the hyper parameters
    hpo = pl.read_csv(LLM_HPO_path)
    mAP = pl.read_csv(LLM_mAP_path)

    combined = hpo.join(mAP, how="left", on="trial")

    # derive the trials remaining
    trials_remaining = 30 - (trial + 1)

    # find whichever trial is the best
    best_run = mAP.filter(pl.col("mean_mAP_50_95").eq(pl.col("mean_mAP_50_95").max()))
    best_trial = best_run["trial"][0]
    best_mAP = best_run["mean_mAP_50_95"][0]

    # find the best hyperparameters as a string
    best_params = ", ".join(f"{key}={value}" for key, value in return_hpo(hpo, best_trial).items())

    historic = []

    # generate the historic values
    for row in combined.iter_rows(named=True):

        trial_num = row["trial"]

        trial_mAP = row["mean_mAP_50_95"]

        # define the hyperparameters for this trial
        params = ", ".join(f"{key}={value}" for key, value in return_hpo(hpo, trial_num).items())


        # append to the list
        historic.append(f"Trial {trial_num}, mean mAP50-95 = {trial_mAP}" + "\n" + "analysis: " + row["analysis"] + "\n" + params)
    
    historic = "\n".join(historic)

    return (loop_prompt
        .replace("{{TRIAL_NUMBER}}", str(trial + 1))
        .replace("{{TRIALS_REMAINING}}", str(trials_remaining))
        .replace("{{TRIAL_HISTORY}}",
                 historic)
        .replace("{{BEST_SCORE}}", str(best_mAP))
        .replace("{{BEST_TRIAL_NUMBER}}", str(best_trial))
        .replace("{{BEST_PARAMS}}",
                 best_params)
    )


In [ ]:
def process_LMM_response(response:dict, trial:int):
    return pl.DataFrame({   
        "trial": trial,
        "analysis": response["analysis"],
        **{key + "_just": value["justification"] for key, value in response["hyperparameters"].items()},
        **{key + "_value": value["value"] for key, value in response["hyperparameters"].items()},

    })

In [ ]:
# define the file paths for the HPOs and the map csv
LLM_HPO_path = Path("LLM_HPO.csv")

LLM_mAP_path = Path("LLM_mAP.csv")

In [ ]:
# loop through 30 trials
for i in range(30):

    # determine if there are prior trials to start from 
    nyc_time = datetime.now(ZoneInfo("America/New_York"))
    print(f"Starting trial {i+1}/30  {nyc_time.strftime("%Y-%m-%d %H:%M:%S %Z")}")

    if i == 0:
        # call the LLM for the first set of hyper parameters
        

        # check if a historic trial exists
        response = load_hpos(i, LLM_HPO_path)
        
        if response is None:
            print("calling LLM")
            response = process_LMM_response(call_fable_5(base_prompt + first_loop_prompt), i)

            # save the CSV
            upsert_unique_csv(response, LLM_HPO_path)

            response = return_hpo(response, i)

        # run the model
        result = load_mAPs(i, LLM_mAP_path)
        if result is None:
            print("running CNN")
            result = LLM_objective(i, response)

            print(result)

            # save the results
            upsert_unique_csv(result, LLM_mAP_path)
    else:

        # derive the prompt
        populated_prompt = populate_loop_prompt(i, LLM_HPO_path, LLM_mAP_path)

        # call the LLM for the first set of hyper parameters
        
        # check if a historic trial exists
        response = load_hpos(i, LLM_HPO_path)
        
        if response is None:
            print("calling LLM")
            response = process_LMM_response(call_fable_5(base_prompt + populated_prompt), i)

            # save the CSV
            upsert_unique_csv(response, LLM_HPO_path)

            response = return_hpo(response, i)

        # run the model
        result = load_mAPs(i, LLM_mAP_path)
        if result is None:
            print("running CNN")
            result = LLM_objective(i, response)

            print(result)

            # save the results
            upsert_unique_csv(result, LLM_mAP_path)

        
        



# LLM test
Note, YOLO requires a valid path for both the training and validation datasets, but since we are training the YOLO model on all of the training data and testing it on the test data, there is no validation data. To get around this, manually edit the YAML file so that the validation path is the same as the training path.

In [ ]:
# generate the run folder name
LLM_run_folder = "LLM_all_train"

# generate the folder, receive the names
image_train_path, image_val_path, label_train_path, label_val_path = make_fold_folder(LLM_run_folder)

# copy over the data to the folders
shift(train, image_train_path, image_val_path, label_train_path, label_val_path, [0,1,2], [])

# make the yaml
make_yaml(LLM_run_folder, image_train_path, image_val_path)

# append the yaml
LLM_yaml = f"{LLM_run_folder}.yaml"

# NOTE: MANUALLY HAVE TO EDIT YAML TO HAVE SAME PATH FOR TRAIN AND VALIDATION SO THAT YOLO IS SATISFIED THAT VALIDATION PATH IS VALID EVEN IF IT DOES NOT USE IT

In [ ]:
LLM_parameters =  {
        'lr0': 0.00214018635106669,
        'lrf': 0.7748126790331027,
        'momentum': 0.8421132632128773, 
        'weight_decay': 0.00039198918876708825,
        'warmup_epochs': 0.9039427650470803,
        'warmup_momentum': 0.7474088093561146,
        **return_hpo(pl.read_csv("LLM_HPO.csv"), 29),
        "val": False
    }

# turn on the yolo print statements
LOGGER.setLevel(logging.INFO)

model = YOLO("yolo26n.pt")

results = model.train(
    data=LLM_yaml,
    epochs=100,
    imgsz=640,
    batch=24,
    device=0,
    optimizer="SGD",
    seed=seed,
    patience=50,
    deterministic=True,
    project="runs/LLM_all",
    name=f"LLM_all",
    verbose=True,
    plots=False,
    **LLM_parameters
)

In [ ]:
model = YOLO(here("runs", "detect", "runs", "LLM_all", "LLM_al", "weights", "last.pt"))

metrics_LLM = model.val(
    data="TEST.yaml",
    split="test",
    imgsz=640,
    batch=24,
    device=0,
    plots=True,
    save_json=True,
    save_txt=True,
    save_conf=True,
)

In [ ]:
box_LLM = metrics_LLM.box

print("Mean precision:", box_LLM.mp)
print("Mean recall:", box_LLM.mr)
print("mAP50:", box_LLM.map50)
print("mAP75:", box_LLM.map75)
print("mAP50-95:", box_LLM.map)
print("Fitness:", metrics_LLM.fitness)